In [1]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import pandas as pd
# Import de ta classe depuis le fichier data_processor.py
from tools.OI_class_v2 import OI_DataProcessor
from tools.OI_class import DataProcessor

# Configuration de l'affichage pour voir toutes les colonnes
pd.set_option('display.max_columns', None)

In [2]:
tags = [ {'tag':'CTY_A1000M_Poids container', 'nom':'A1000M_Poids' },
         {'tag' : 'CTY_A1000M_Teneur arr. Vit. A (UV)', 'nom':'A1000M_UV_Auto' },
         {'tag' :'CTY_A1000M_Titre VA AC', 'nom':'A1000M_Labo' },
         {'tag' :'CTY_A1000M_Numéro Container', 'nom':'A1000M_Num' },
         {'tag' :'CTY_A1000R_Poids container', 'nom':'A1000R_Poids' },
         {'tag' :'CTY_A1000R_Teneur arr. Vit. A (UV)', 'nom':'A1000R_UV_Auto' },
         {'tag' :'CTY_A1000R_Titre VA AC', 'nom':'A1000R_Labo' },
         {'tag' :'CTY_A1000R_Numéro Container', 'nom':'A1000R_Num' } ]

In [ ]:
# Initialisation
processor = OI_DataProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    tags_other = [],
    tags_selected = tags,
    start='2025-01-01',
    end='2025-12-18',
    interval='PT20M',
    verbose=False
)

# Charger les données
processor.merge()

# Appliquer des filtres (chaînage possible)
processor.filtering(
    tag=['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo'],
    min_val=[700, 800_000, 700_000],
    max_val=[1500, 1_300_000, 1_300_000],
    na=['A1000M_Poids','A1000R_Poids']
)

# Ajouter des colonnes
processor.ajoute_cumul('A1000M_Poids', 'A1000M_UV_Auto', 1000, 'Cumul_UV')
processor.ajoute_cumul('A1000M_Poids', 'A1000M_Labo', 1000, 'Cumul_Labo')
processor.ajouter_moyennes_glissantes('A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Moyenne_UV', window=10)
processor.ajouter_moyennes_glissantes('A1000M_Poids', 'A1000M_Labo', 'A1000M_Moyenne_Labo', window=10)

# Afficher les infos finales
processor.info()

INFORMATIONS DataProcessor
URL de base       : https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?
Tags autres       : []
Tags sélectionnés : [{'tag': 'CTY_A1000M_Poids container', 'nom': 'A1000M_Poids'}, {'tag': 'CTY_A1000M_Teneur arr. Vit. A (UV)', 'nom': 'A1000M_UV_Auto'}, {'tag': 'CTY_A1000M_Titre VA AC', 'nom': 'A1000M_Labo'}, {'tag': 'CTY_A1000M_Numéro Container', 'nom': 'A1000M_Num'}, {'tag': 'CTY_A1000R_Poids container', 'nom': 'A1000R_Poids'}, {'tag': 'CTY_A1000R_Teneur arr. Vit. A (UV)', 'nom': 'A1000R_UV_Auto'}, {'tag': 'CTY_A1000R_Titre VA AC', 'nom': 'A1000R_Labo'}, {'tag': 'CTY_A1000R_Numéro Container', 'nom': 'A1000R_Num'}]
Tous les tags API : ['CTY_A1000M_Poids container', 'CTY_A1000M_Teneur arr. Vit. A (UV)', 'CTY_A1000M_Titre VA AC', 'CTY_A1000M_Numéro Container', 'CTY_A1000R_Poids container', 'CTY_A1000R_Teneur arr. Vit. A (UV)', 'CTY_A1000R_Titre VA AC', 'CTY_A1000R_Numéro Container']
Mapping renommage : {'CTY_A1000M_Poids container': 'A1000M_Po

In [12]:

# 2. On veut changer de date ?
processor.start = '2010-01-01'
#processor.set_end('2022-01-15')

# 3. On recalcule TOUT (merge + filtres + cumul) en une commande
processor.recalculate()

# 4. On vérifie le pipeline
processor.show_pipeline()


--- PIPELINE ACTUEL ---
1. filtering | Args: () | Kwargs: {'tag': ['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo'], 'min_val': [700, 800000, 700000], 'max_val': [1500, 1300000, 1300000], 'na': 'A1000M_Poids'}
2. ajoute_cumul | Args: ('A1000M_Poids', 'A1000M_UV_Auto', 1000, 'Cumul_UV') | Kwargs: {}
3. ajoute_cumul | Args: ('A1000M_Poids', 'A1000M_Labo', 1000, 'Cumul_Labo') | Kwargs: {}
4. ajouter_moyennes_glissantes | Args: ('A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Moyenne_UV') | Kwargs: {'window': 10}
5. ajouter_moyennes_glissantes | Args: ('A1000M_Poids', 'A1000M_Labo', 'A1000M_Moyenne_Labo') | Kwargs: {'window': 10}
-----------------------



In [7]:
processor.data.describe()

,A1000M_Poids,A1000M_UV_Auto,A1000M_Labo,A1000M_Num,A1000R_Poids,A1000R_UV_Auto,A1000R_Labo,A1000R_Num,Cumul_UV,Cumul_Labo,A1000M_Moyenne_UV,A1000M_Moyenne_Labo
count,6778.000000,6.778000e+03,6.778000e+03,6525.000000,1.0,6.778000e+03,6.778000e+03,1.0,6.778000e+03,6.778000e+03,6.769000e+03,6.769000e+03
mean,1080.813268,1.030306e+06,1.031765e+06,94.005223,1027.0,1.027478e+06,1.039397e+06,152.0,1.113579e+06,1.115170e+06,1.030289e+06,1.031776e+06
std,72.086828,2.671958e+04,2.564269e+04,64.992387,NaN,2.817671e+04,2.157732e+04,NaN,7.985193e+04,7.971248e+04,2.364826e+04,2.093237e+04
min,703.000000,8.690000e+05,7.562306e+05,1.000000,1027.0,9.771167e+05,1.003422e+06,152.0,7.353380e+05,7.030000e+05,9.040000e+05,9.428494e+05
25%,1036.000000,1.016000e+06,1.015650e+06,39.000000,1027.0,1.007000e+06,1.029093e+06,152.0,1.062786e+06,1.063155e+06,1.016704e+06,1.018021e+06
50%,1086.000000,1.031000e+06,1.031989e+06,89.000000,1027.0,1.026000e+06,1.037800e+06,152.0,1.116891e+06,1.119919e+06,1.031000e+06,1.031241e+06
75%,1129.000000,1.047000e+06,1.047498e+06,135.000000,1027.0,1.040400e+06,1.057600e+06,152.0,1.168270e+06,1.169154e+06,1.045212e+06,1.044510e+06
max,1295.000000,1.113000e+06,1.171338e+06,1345.000000,1027.0,1.121000e+06,1.098000e+06,152.0,1.387825e+06,1.418491e+06,1.113000e+06,1.115780e+06


In [22]:
processor.df['CTY_A1000R_Poids container'].describe()

count     125.000000
mean     1106.544000
std        71.067069
min       919.000000
25%      1059.500000
50%      1109.000000
75%      1160.000000
max      1256.000000
Name: CTY_A1000R_Poids container, dtype: float64

In [16]:
processor.plot_tag(tag=['A1000R_Poids','A1000R_UV_Auto','A1000R_Labo'])